# Llama-3.1-8B-Instruct — Probe-1 greedy behavioural (Colab T4)

**Model:** `meta-llama/Llama-3.1-8B-Instruct` · **8-bit (int8 bitsandbytes)** · **greedy** (`do_sample=False`).

Runs Probe-1 **canonical and W3** for GSM, ALGO, and BW using:

1. The **Appendix N** Probe-1 template (paper `\label{app:prompts}`).
2. The **released verifiers** in `probes/` — imported, not reimplemented.
3. The **OpenRouter answer-extraction rule**: GSM prefers `#### <num>`, else the **last** numeric token (`verify_gsm_answer`). Appendix O / the local-harness bug is routing GSM through `verify_answer(..., family="gsm")`, which uses **first-number** extraction (`_verify_numeric`) and scores 0/44.

Output: `colab_out/llama_greedy_p1.csv` plus a Table-7 comparison.

Greedy is deterministic, so 3 seeds are N/A. A second pass draws **3 samples at T=1.0** on a stratified subset to quantify decoding variance.

**Secrets (Colab → 🔑):** `HF_TOKEN` (gated Llama), optional `GITHUB_TOKEN` if the repo is private.


In [ ]:
# Colab T4: bitsandbytes for quantized loads. Restart the runtime if
# bitsandbytes was just installed and the kernel has not picked it up.
import sys
import subprocess
subprocess.check_call(
    [
        sys.executable, "-m", "pip", "install", "-q", "-U",
        "transformers>=4.44",
        "accelerate>=0.33",
        "bitsandbytes>=0.43",
        "pandas",
        "scipy",
        "networkx",
        "tqdm",
        "huggingface_hub",
    ]
)


In [ ]:
from __future__ import annotations

import os
import subprocess
import sys
from pathlib import Path

# ── knobs ────────────────────────────────────────────────────────────────
# Set LIMIT to an int for a smoke test (e.g. 2 items per family). None = full run.
LIMIT = None
DRY_RUN = False          # True: skip GPU, write placeholder rows (pipeline check)
RESUME = True

# Private GitHub clone (Colab secret GITHUB_TOKEN, or env). Public clone works
# without a token. If this notebook is already inside the repo, clone is skipped.
REPO_URL = os.environ.get(
    "RVC_REPO_URL",
    "https://github.com/Adya6714/retrieval-vs-computation.git",
)
REPO_COMMIT = os.environ.get("RVC_REPO_COMMIT", "")  # empty = default branch HEAD

def _secret(name: str) -> str:
    v = os.environ.get(name, "")
    if v:
        return v
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get(name) or ""
    except Exception:
        return ""

HF_TOKEN = _secret("HF_TOKEN") or _secret("HUGGING_FACE_HUB_TOKEN")
GH_TOKEN = _secret("GITHUB_TOKEN")

# Llama-3.1-8B-Instruct is gated: https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    try:
        from huggingface_hub import login as _hf_login
        _hf_login(token=HF_TOKEN, add_to_git_credential=False)
    except Exception as _hf_exc:
        print("[setup] huggingface login skipped:", _hf_exc)

def _looks_like_repo(p: Path) -> bool:
    return (p / "probes" / "contamination" / "verify.py").is_file() and (
        p / "data" / "problems" / "question_bank_gsm.csv"
    ).is_file()

def _find_repo() -> Path:
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if _looks_like_repo(cand):
            return cand
    colab = Path("/content/retrieval-vs-computation")
    if _looks_like_repo(colab):
        return colab
    return colab

REPO_ROOT = _find_repo()
if not _looks_like_repo(REPO_ROOT):
    REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
    url = REPO_URL
    if GH_TOKEN and "github.com" in url and url.startswith("https://"):
        url = url.replace("https://", f"https://{GH_TOKEN}@")
    print(f"[setup] cloning {REPO_URL} → {REPO_ROOT}")
    cmd = ["git", "clone", "--depth", "1", url, str(REPO_ROOT)]
    subprocess.check_call(cmd)
    if REPO_COMMIT:
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", REPO_COMMIT])
        subprocess.check_call(["git", "-C", str(REPO_ROOT), "checkout", REPO_COMMIT])

assert _looks_like_repo(REPO_ROOT), (
    f"Could not find probes/ + question banks under {REPO_ROOT}. "
    "Clone the retrieval-vs-computation repo, or set RVC_REPO_URL / GITHUB_TOKEN."
)
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

OUT_DIR = Path("/content/colab_out") if Path("/content").exists() else (REPO_ROOT / "colab_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"[setup] REPO_ROOT={REPO_ROOT}")
print(f"[setup] OUT_DIR={OUT_DIR}")
print(f"[setup] LIMIT={LIMIT} DRY_RUN={DRY_RUN} RESUME={RESUME}")


## Appendix O item 7 — extraction rule (must pass before any scoring)

OpenRouter P1 GSM uses `verify_gsm_answer` (via `family="arithmetic_reasoning"`).

Passing `family="gsm"` hits `_verify_numeric`, which takes the **first** number in the string. That is the documented local-harness bug.


In [ ]:
import re
from probes.contamination.verify import verify_answer, verify_gsm_answer, _verify_numeric
from probes.contamination.verify_algo import verify_algo

# --- Appendix O item 7: first-number vs last-number / #### -----------------
BUG_RESPONSE = (
    "The hotel charges 0.9 first. Working: 20*0.9=18. Then #### 51"
)
LAST_ONLY = "step1=12 step2=18 the cost is 51"
FIRST_WRONG = "12 is an intermediate; the answer is 51"

assert verify_gsm_answer(BUG_RESPONSE, "51") is True, "#### tag must win"
assert verify_gsm_answer(LAST_ONLY, "51") is True, "last number must win when no ####"
assert verify_gsm_answer(FIRST_WRONG, "51") is True

# family='gsm' is the bug path (first number: 0.9 or 12)
assert verify_answer("GSM_001", BUG_RESPONSE, "51", "gsm") is False, (
    "family='gsm' must NOT be used: it is first-number extraction"
)
assert verify_answer("GSM_001", LAST_ONLY, "51", "gsm") is False
assert _verify_numeric(LAST_ONLY, "51") is False, "first number of LAST_ONLY is 12, not 51"

# OpenRouter / bank routing: arithmetic_reasoning → verify_gsm_answer
assert verify_answer("GSM_001", BUG_RESPONSE, "51", "arithmetic_reasoning") is True
assert verify_answer("GSM_001", LAST_ONLY, "51", "arithmetic_reasoning") is True

print("PASS: GSM scoring uses #### / last-number (OpenRouter). First-number path is asserted-against.")
print("  verify_gsm_answer(BUG_RESPONSE, 51) =", verify_gsm_answer(BUG_RESPONSE, "51"))
print("  verify_answer(..., family='gsm')     =", verify_answer("GSM_001", BUG_RESPONSE, "51", "gsm"), "  ← bug")
print("  verify_answer(..., arithmetic_reasoning) =", verify_answer("GSM_001", BUG_RESPONSE, "51", "arithmetic_reasoning"))


## Prompt (Appendix N) and item queue

Paper Appendix (Prompts):

> Solve the following problem exactly and provide only the final answer in the required output format. Problem: `{problem}`. Format instruction: `{family_specific_output_format}`.

The same template is used for canonical and W3; only `{problem}` changes. Instruct models still need the chat template around that user string.


In [ ]:
import csv
import json
import random
from typing import Any

import pandas as pd
from tqdm.auto import tqdm

# Appendix N Probe-1 template (paper/appendix.tex \label{app:prompts})
PROBE1_TEMPLATE = (
    "Solve the following problem exactly and provide only the final answer "
    "in the required output format. Problem: {problem}. Format instruction: "
    "{family_specific_output_format}."
)

FAMILY_FORMAT = {
    "GSM": (
        "Write the final numerical answer on its own line as #### <number>. "
        "No other text after that tag."
    ),
    "ALGO": (
        "Follow the problem's required output format exactly "
        "(Path: / Count: / Selected: or Total: / Scoops:). No explanation."
    ),
    "BW": (
        "A numbered list of actions only. Each action must be one of the "
        "permitted operators with their arguments. No explanation."
    ),
}

TABLE7_LLAMA = {
    # Table 7 Probe-1 Llama cells (OpenRouter). GSM n=20 (GSM_001–020). BW n=65.
    ("GSM", "Can."): 0.800,
    ("GSM", "W3"): 0.150,
    ("BW", "Can."): 0.015,
    ("BW", "W3"): 0.108,
}


def _norm_vt(v: str) -> str:
    v = str(v).strip()
    return "canonical" if v.lower() == "canonical" else v.upper()


def _strip_csv_quotes(text: str) -> str:
    s = str(text)
    if len(s) >= 2 and s[0] == '"' and s[-1] == '"':
        s = s[1:-1]
    return s


def _load_bank(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str).fillna("")
    df["problem_id"] = df["problem_id"].astype(str).str.strip()
    df["variant_type"] = df["variant_type"].map(_norm_vt)
    df["problem_text"] = df["problem_text"].map(_strip_csv_quotes)
    return df


def build_prompt(problem_text: str, family: str) -> str:
    return PROBE1_TEMPLATE.format(
        problem=problem_text.strip(),
        family_specific_output_format=FAMILY_FORMAT[family],
    )


def load_items(limit: int | None) -> list[dict[str, Any]]:
    """Canonical + W3 for all three families (paired IDs only)."""
    specs = [
        ("GSM", REPO_ROOT / "data/problems/question_bank_gsm.csv"),
        ("ALGO", REPO_ROOT / "data/problems/question_bank_algo.csv"),
        ("BW", REPO_ROOT / "data/problems/question_bank_bw.csv"),
    ]
    items: list[dict[str, Any]] = []
    for family, path in specs:
        df = _load_bank(path)
        can_ids = set(df.loc[df.variant_type == "canonical", "problem_id"])
        w3_ids = set(df.loc[df.variant_type == "W3", "problem_id"])
        paired = sorted(can_ids & w3_ids)
        if family == "GSM":
            # 44 bank IDs (same universe as GSM_P1_behavioral_claude.csv)
            pass
        n_take = paired if limit is None else paired[:limit]
        for pid in n_take:
            for vt in ("canonical", "W3"):
                row = df[(df.problem_id == pid) & (df.variant_type == vt)].iloc[0]
                items.append(
                    {
                        "problem_id": pid,
                        "family": family,
                        "variant": vt,
                        "problem_text": str(row["problem_text"]),
                        "correct_answer": str(row["correct_answer"]),
                        "problem_subtype": str(row.get("problem_subtype", "")).strip().lower(),
                        "difficulty_params": str(row.get("difficulty_params", "{}") or "{}"),
                    }
                )
    return items


def score_item(item: dict, model_answer: str) -> bool:
    """Import-only scoring. GSM must never go through family='gsm'."""
    fam = item["family"]
    if fam == "GSM":
        return bool(verify_gsm_answer(model_answer, item["correct_answer"]))
    if fam == "ALGO":
        ok, _reason, _meta = verify_algo(
            item["problem_id"],
            model_answer,
            item["correct_answer"],
            item["problem_subtype"],
            item["variant"],
            item["difficulty_params"],
        )
        return bool(ok)
    # BW / Mystery BW
    vf = (
        "mystery_blocksworld"
        if item["problem_subtype"] == "mystery_blocksworld"
        or str(item["problem_id"]).startswith("MBW_")
        else "blocksworld"
    )
    return bool(
        verify_answer(
            item["problem_id"],
            model_answer,
            item["correct_answer"],
            vf,
            problem_text=item["problem_text"],
        )
    )


ITEMS = load_items(LIMIT)
print(f"[queue] {len(ITEMS)} prompts")
print(pd.DataFrame(ITEMS).groupby(["family", "variant"]).size().to_string())


## Hugging Face login (gated Llama)

Run this **before** the model-load cell. Uses `HF_TOKEN` from Colab secrets if set; otherwise prompts.


In [ ]:
from huggingface_hub import login, get_token

_tok = HF_TOKEN or get_token()
if _tok:
    login(token=_tok, add_to_git_credential=False)
    print("[hf] authenticated")
else:
    login()


## Google Drive: restore `llama_greedy_p1.csv` if this runtime has no copy

Copies only. Does not use the GPU or rerun inference. Restores from
`MyDrive/llama_outputs/llama_greedy_p1.csv` so `RESUME=True` can skip completed rows.


In [ ]:
import shutil
from pathlib import Path

GREEDY_CSV = Path("/content/colab_out/llama_greedy_p1.csv") if Path("/content").exists() else (OUT_DIR / "llama_greedy_p1.csv")
DRIVE_CSV = Path("/content/drive/MyDrive/llama_outputs/llama_greedy_p1.csv")

if Path("/content").exists() and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print("[drive] mount skipped:", exc)

OUT_DIR.mkdir(parents=True, exist_ok=True)
if (not GREEDY_CSV.exists() or GREEDY_CSV.stat().st_size == 0) and DRIVE_CSV.exists():
    shutil.copy2(DRIVE_CSV, GREEDY_CSV)
    print(f"[restore] {DRIVE_CSV} -> {GREEDY_CSV}  ({GREEDY_CSV.stat().st_size} bytes)")
elif GREEDY_CSV.exists():
    print(f"[restore] local CSV already present: {GREEDY_CSV}  ({GREEDY_CSV.stat().st_size} bytes)")
else:
    print(f"[restore] no local or Drive CSV yet; greedy sweep will write {GREEDY_CSV}")


## Load Llama-3.1-8B-Instruct in 8-bit (T4 16GB)

Skip this cell if `llama_greedy_p1.csv` is already recovered — the sweep resumes completed rows and the summary does not need the GPU.


In [ ]:
import gc
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
COMPUTE_DTYPE = torch.float16  # T4 has no native bfloat16

# GSM was 128; 23/37 wrong canonical answers in llama_greedy_p1.csv end mid-token
# (correct median 74 chars, incorrect median 444, max 552). G4 reruns at 768.
MAX_NEW = {"GSM": 768, "ALGO": 192, "BW": 512}

tokenizer = None
model = None
DEVICE = "cpu"

if not DRY_RUN:
    assert torch.cuda.is_available(), "This notebook expects a GPU (Colab T4)."
    print("[gpu]", torch.cuda.get_device_name(0), "mem_GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    bnb_config = BitsAndBytesConfig(load_in_8bit=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=HF_TOKEN or True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=COMPUTE_DTYPE,
        token=HF_TOKEN or True,
    )
    model.eval()
    DEVICE = next(model.parameters()).device
    print("[model] loaded 8-bit", MODEL_ID, "device", DEVICE)
else:
    print("[dry-run] skipping model load")


def wrap_chat(user_text: str) -> str:
    assert tokenizer is not None
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": user_text}],
        add_generation_prompt=True,
        tokenize=False,
    )


@torch.inference_mode()
def generate(user_text: str, family: str, *, do_sample: bool, temperature: float | None = None) -> str:
    if DRY_RUN:
        return "#### 0" if family == "GSM" else "DRY_RUN"
    prompt = wrap_chat(user_text)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    gen_kwargs: dict = dict(
        max_new_tokens=MAX_NEW[family],
        pad_token_id=tokenizer.pad_token_id,
        do_sample=do_sample,
    )
    if do_sample:
        gen_kwargs["temperature"] = float(temperature if temperature is not None else 1.0)
        gen_kwargs["top_p"] = 1.0
    out = model.generate(**inputs, **gen_kwargs)
    new = out[0, inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new, skip_special_tokens=True).strip()


## Greedy sweep → `colab_out/llama_greedy_p1.csv`

`RESUME=True` skips completed `(problem_id, family, variant)` rows in this file.
That **does not** re-run truncated GSM canonical answers (25/37 wrong answers
are exactly 128 tokens). The 768-token GSM canonical rerun is a **later cell**
and writes a **new** CSV. Never overwrite `llama_greedy_p1.csv` with
the 768 run.


In [ ]:
GREEDY_CSV = OUT_DIR / "llama_greedy_p1.csv"
GREEDY_COLS = ["problem_id", "family", "variant", "model_answer", "correct"]

def _done_keys(path: Path) -> set[tuple[str, str, str]]:
    if not (RESUME and path.exists() and path.stat().st_size > 0):
        return set()
    prev = pd.read_csv(path, dtype=str)
    return {
        (str(r.problem_id), str(r.family), str(r.variant))
        for _, r in prev.iterrows()
    }

done = _done_keys(GREEDY_CSV)
write_header = not GREEDY_CSV.exists() or GREEDY_CSV.stat().st_size == 0
if not RESUME and GREEDY_CSV.exists():
    GREEDY_CSV.unlink()
    write_header = True
    done = set()

n_ok = n_done = 0
if RESUME and GREEDY_CSV.exists() and GREEDY_CSV.stat().st_size > 0:
    prev = pd.read_csv(GREEDY_CSV, dtype=str)
    n_done = len(prev)
    n_ok = int(prev["correct"].astype(str).str.lower().isin(["true", "1"]).sum())

with GREEDY_CSV.open("a", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=GREEDY_COLS)
    if write_header:
        w.writeheader()
    for item in tqdm(ITEMS, desc="greedy"):
        key = (item["problem_id"], item["family"], item["variant"])
        if key in done:
            continue
        user = build_prompt(item["problem_text"], item["family"])
        ans = generate(user, item["family"], do_sample=False)
        correct = bool(score_item(item, ans))
        w.writerow(
            {
                "problem_id": item["problem_id"],
                "family": item["family"],
                "variant": item["variant"],
                "model_answer": ans,
                "correct": correct,
            }
        )
        f.flush()
        n_done += 1
        n_ok += int(correct)
        done.add(key)

print(f"wrote {GREEDY_CSV}  running_acc={n_ok}/{n_done}")
greedy_df = pd.read_csv(GREEDY_CSV, dtype=str)
greedy_df["correct_bool"] = greedy_df["correct"].astype(str).str.lower().isin(["true", "1"])
print(greedy_df.groupby(["family", "variant"])["correct_bool"].agg(["mean", "sum", "count"]))


## Google Drive backup (copy only — no GPU, no inference)


In [ ]:
from pathlib import Path
import shutil

src = Path("/content/colab_out/llama_greedy_p1.csv")
drive_dir = Path("/content/drive/MyDrive/llama_outputs")

if Path("/content").exists() and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print("[drive] mount skipped:", exc)

drive_dir.mkdir(parents=True, exist_ok=True)

dst = drive_dir / "llama_greedy_p1.csv"

if src.exists():
    shutil.copy2(src, dst)
    print(f"[backup] copied {src} -> {dst}  ({dst.stat().st_size} bytes)")
else:
    print(f"[backup] source not found: {src}")

# Also pull the files onto your laptop (Colab → browser download).
try:
    from google.colab import files as _colab_files  # type: ignore
    for p in (src, Path("/content/colab_out/llama_greedy_p1_manifest.json")):
        if p.exists():
            _colab_files.download(str(p))
            print(f"[download] {p.name}")
except Exception as exc:
    print("[download] skipped (not Colab or download blocked):", exc)


## H6 — GSM canonical greedy rerun at 768 tokens (new file)

The original `llama_greedy_p1.csv` GSM canonical cell is 7/44 = 0.159 because
generation was capped at 128 tokens. This cell reruns **GSM canonical only**
at `max_new_tokens=768` and writes `llama_greedy_p1_gsm_canonical_768.csv`.
It does **not** read or write `llama_greedy_p1.csv`, so `RESUME=True` on the
truncated file cannot skip these rows.

Requires GPU. Copy the new CSV into `results/raw/llama_greedy_p1_gsm_canonical_768.csv`
after the run. Until it exists, there is no local-vs-OpenRouter GSM greedy comparison.


In [ ]:
from pathlib import Path
import csv

GSM_768_CSV = OUT_DIR / "llama_greedy_p1_gsm_canonical_768.csv"
if GSM_768_CSV.resolve() == (OUT_DIR / "llama_greedy_p1.csv").resolve():
    raise SystemExit("Refusing to overwrite llama_greedy_p1.csv")

gsm_can_items = [
    it for it in ITEMS
    if it["family"] == "GSM" and str(it["variant"]).lower() == "canonical"
]
print(f"[H6] GSM canonical queue n={len(gsm_can_items)}  out={GSM_768_CSV}")

def _done_768(path: Path) -> set[str]:
    if not (path.exists() and path.stat().st_size > 0):
        return set()
    prev = pd.read_csv(path, dtype=str)
    return set(prev["problem_id"].astype(str))

done768 = _done_768(GSM_768_CSV)
write_header768 = not GSM_768_CSV.exists() or GSM_768_CSV.stat().st_size == 0
n_ok768 = n_done768 = 0
if GSM_768_CSV.exists() and GSM_768_CSV.stat().st_size > 0:
    prev = pd.read_csv(GSM_768_CSV, dtype=str)
    n_done768 = len(prev)
    n_ok768 = int(prev["correct"].astype(str).str.lower().isin(["true", "1"]).sum())

COLS768 = ["problem_id", "family", "variant", "model_answer", "correct", "max_new_tokens"]
with GSM_768_CSV.open("a", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=COLS768)
    if write_header768:
        w.writeheader()
    for item in tqdm(gsm_can_items, desc="gsm_canonical_768"):
        pid = str(item["problem_id"])
        if pid in done768:
            continue
        user = build_prompt(item["problem_text"], "GSM")
        ans = generate(user, "GSM", do_sample=False)
        correct = bool(score_item(item, ans))
        w.writerow(
            {
                "problem_id": pid,
                "family": "GSM",
                "variant": "canonical",
                "model_answer": ans,
                "correct": correct,
                "max_new_tokens": 768,
            }
        )
        f.flush()
        n_done768 += 1
        n_ok768 += int(correct)
        done768.add(pid)

print(f"wrote {GSM_768_CSV}  running_acc={n_ok768}/{n_done768}")
if GSM_768_CSV.exists() and GSM_768_CSV.stat().st_size > 0:
    g768 = pd.read_csv(GSM_768_CSV, dtype=str)
    g768["correct_bool"] = g768["correct"].astype(str).str.lower().isin(["true", "1"])
    print(g768["correct_bool"].agg(["mean", "sum", "count"]))


## Google Drive backup of the 768 GSM CSV (copy only)


In [ ]:
from pathlib import Path
import shutil

src768 = OUT_DIR / "llama_greedy_p1_gsm_canonical_768.csv"
drive_dir = Path("/content/drive/MyDrive/llama_outputs")
if Path("/content").exists() and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print("[drive] mount skipped:", exc)
drive_dir.mkdir(parents=True, exist_ok=True)
if src768.exists():
    dst768 = drive_dir / "llama_greedy_p1_gsm_canonical_768.csv"
    shutil.copy2(src768, dst768)
    print(f"[backup] copied {src768} -> {dst768}  ({dst768.stat().st_size} bytes)")
    try:
        from google.colab import files as _colab_files  # type: ignore
        _colab_files.download(str(src768))
        print(f"[download] {src768.name}")
    except Exception as exc:
        print("[download] skipped:", exc)
else:
    print(f"[backup] 768 GSM CSV not found: {src768} (run the H6 cell on a GPU runtime)")


## Summary vs Table 7 (Llama OpenRouter)

Table 7 GSM Llama is **n=20** (GSM_001–020). This notebook scores the full **n=44** bank; both slices are printed. ALGO Table 7 is per subtype×instance-type — we print pooled can/W3 plus a subtype breakdown when `difficulty_params` is present.


In [ ]:
def acc_table(df: pd.DataFrame) -> pd.DataFrame:
    g = (
        df.groupby(["family", "variant"], as_index=False)["correct_bool"]
        .agg(n="count", n_correct="sum", acc="mean")
    )
    return g.sort_values(["family", "variant"])

summary = acc_table(greedy_df)
print("=== Local greedy (this notebook) ===")
print(summary.to_string(index=False))

print("\n=== vs Table 7 Llama OpenRouter (Can / W3) ===")
rows = []
for fam in ("GSM", "ALGO", "BW"):
    sub = greedy_df[greedy_df.family == fam]
    for vt, t7k in (("canonical", "Can."), ("W3", "W3")):
        s = sub[sub.variant == vt]
        local = float(s.correct_bool.mean()) if len(s) else float("nan")
        t7 = TABLE7_LLAMA.get((fam, t7k), float("nan"))
        rows.append(
            {
                "family": fam,
                "variant": vt,
                "n_local": len(s),
                "acc_local_greedy": None if pd.isna(local) else round(local, 3),
                "acc_table7_llama_openrouter": None if pd.isna(t7) else t7,
            }
        )
print(pd.DataFrame(rows).to_string(index=False))

# GSM bank-valid n=20 slice (matches Table 7 denominator)
gsm20 = greedy_df[
    (greedy_df.family == "GSM")
    & (greedy_df.problem_id.str.extract(r"GSM_(\d+)", expand=False).astype(float) <= 20)
]
if len(gsm20):
    print("\n=== GSM_001–020 only (Table 7 n=20) ===")
    print(acc_table(gsm20).to_string(index=False))

# ALGO subtype breakdown (Table 7 slices)
algo_items = { (it["problem_id"], it["variant"]): it for it in ITEMS if it["family"] == "ALGO" }
if algo_items:
    parts = []
    for _, r in greedy_df[greedy_df.family == "ALGO"].iterrows():
        it = algo_items.get((r.problem_id, r.variant))
        if not it:
            continue
        try:
            dp = json.loads(it["difficulty_params"] or "{}")
        except json.JSONDecodeError:
            dp = {}
        parts.append(
            {
                "subtype": it["problem_subtype"],
                "instance_type": dp.get("instance_type", ""),
                "variant": r.variant,
                "correct_bool": r.correct_bool,
            }
        )
    if parts:
        adf = pd.DataFrame(parts)
        print("\n=== ALGO by subtype × instance_type ===")
        print(
            adf.groupby(["subtype", "instance_type", "variant"])["correct_bool"]
            .agg(["mean", "sum", "count"])
            .to_string()
        )


## Decoding variance — 3 samples at `temperature=1.0`

Greedy has no seed axis. This cell samples the same Appendix-N prompt three times at T=1.0 on a **stratified subset** (2 problem IDs × canonical+W3 × 3 families = 12 items × 3 = 36 generations). Set `FULL_VARIANCE=True` to sample every greedy item (slow on a free T4).


In [ ]:
FULL_VARIANCE = False
VARIANCE_IDS_PER_FAMILY = 2
N_SAMPLES = 3
TEMP = 1.0
VAR_CSV = OUT_DIR / "llama_temp1_variance.csv"
VAR_COLS = [
    "problem_id", "family", "variant", "sample_id", "model_answer", "correct",
]

rng = random.Random(0)
var_items: list[dict] = []
if FULL_VARIANCE:
    var_items = list(ITEMS)
else:
    by_fam: dict[str, list[str]] = {}
    for it in ITEMS:
        if it["variant"] == "canonical":
            by_fam.setdefault(it["family"], []).append(it["problem_id"])
    chosen = []
    for fam, pids in by_fam.items():
        uniq = sorted(set(pids))
        k = min(VARIANCE_IDS_PER_FAMILY, len(uniq))
        sampled = rng.sample(uniq, k) if k else []
        chosen.extend((fam, pid) for pid in sampled)
    want = {(fam, pid) for fam, pid in chosen}
    var_items = [it for it in ITEMS if (it["family"], it["problem_id"]) in want]

print(f"[variance] {len(var_items)} items × {N_SAMPLES} samples at T={TEMP}")

var_header = not VAR_CSV.exists() or VAR_CSV.stat().st_size == 0
var_done = set()
if RESUME and VAR_CSV.exists() and VAR_CSV.stat().st_size > 0:
    vprev = pd.read_csv(VAR_CSV, dtype=str)
    var_done = {
        (str(r.problem_id), str(r.family), str(r.variant), str(r.sample_id))
        for _, r in vprev.iterrows()
    }

with VAR_CSV.open("a", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=VAR_COLS)
    if var_header:
        w.writeheader()
    for item in tqdm(var_items, desc="T=1.0"):
        user = build_prompt(item["problem_text"], item["family"])
        for sid in range(N_SAMPLES):
            key = (item["problem_id"], item["family"], item["variant"], str(sid))
            if key in var_done:
                continue
            ans = generate(user, item["family"], do_sample=True, temperature=TEMP)
            w.writerow(
                {
                    "problem_id": item["problem_id"],
                    "family": item["family"],
                    "variant": item["variant"],
                    "sample_id": sid,
                    "model_answer": ans,
                    "correct": bool(score_item(item, ans)),
                }
            )
            f.flush()

vdf = pd.read_csv(VAR_CSV, dtype=str)
vdf["correct_bool"] = vdf["correct"].astype(str).str.lower().isin(["true", "1"])
# pairwise exact-match of raw strings across the 3 samples
agree_rows = []
for (pid, fam, vt), g in vdf.groupby(["problem_id", "family", "variant"]):
    texts = g.sort_values("sample_id")["model_answer"].astype(str).tolist()
    accs = g.sort_values("sample_id")["correct_bool"].tolist()
    n_unique = len(set(texts))
    agree_rows.append(
        {
            "problem_id": pid,
            "family": fam,
            "variant": vt,
            "n_unique_strings": n_unique,
            "all_three_identical": n_unique == 1,
            "acc_mean": float(sum(accs) / max(len(accs), 1)),
            "acc_min": float(min(accs) if accs else 0),
            "acc_max": float(max(accs) if accs else 0),
        }
    )
adf = pd.DataFrame(agree_rows)
print("\n=== T=1.0 variance ===")
print(f"items={len(adf)}  fraction all-3-identical={adf.all_three_identical.mean():.3f}")
print(adf.groupby("family")[["all_three_identical", "acc_mean", "n_unique_strings"]].mean())
print(f"wrote {VAR_CSV}")


## Manifest


In [ ]:
import subprocess

def git_hash() -> str:
    try:
        return subprocess.check_output(
            ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"], text=True
        ).strip()
    except Exception as exc:
        return f"unavailable ({exc})"

quant = "int8 bitsandbytes" if not DRY_RUN else "DRY_RUN"
dtype = str(COMPUTE_DTYPE) if not DRY_RUN else "n/a"
n_items = int(len(greedy_df.drop_duplicates(["problem_id", "family", "variant"]))) if "greedy_df" in dir() else len(ITEMS)

manifest = {
    "notebook": "llama_greedy_behavioural.ipynb",
    "model_string": MODEL_ID,
    "dtype": dtype,
    "quantization": quant,
    "decoding_config": {
        "greedy": {"do_sample": False, "temperature": None},
        "variance_pass": {
            "do_sample": True,
            "temperature": 1.0,
            "n_samples": 3,
            "full_variance": FULL_VARIANCE,
        },
    },
    "n_items": n_items,
    "n_greedy_rows": int(len(greedy_df)) if "greedy_df" in dir() else None,
    "families": ["GSM", "ALGO", "BW"],
    "variants": ["canonical", "W3"],
    "prompt": "Appendix N Probe-1 + Llama-3.1 Instruct chat template",
    "verifier": "probes.contamination.verify.verify_gsm_answer / verify_answer(blocksworld) / verify_algo",
    "extraction_rule": "GSM: #### tag else last number (OpenRouter). family='gsm' first-number path asserted-against.",
    "git_commit_hash": git_hash(),
    "output_csv": str(GREEDY_CSV),
}
print("=== MANIFEST ===")
print(json.dumps(manifest, indent=2))
(OUT_DIR / "llama_greedy_p1_manifest.json").write_text(json.dumps(manifest, indent=2))

_out_files = [
    OUT_DIR / "llama_greedy_p1.csv",
    OUT_DIR / "llama_greedy_p1_manifest.json",
]
_drive_dir = Path("/content/drive/MyDrive/llama_outputs")
if Path("/content").exists() and not Path("/content/drive/MyDrive").exists():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print("[drive] mount skipped:", exc)
try:
    import shutil as _shutil
    _drive_dir.mkdir(parents=True, exist_ok=True)
    for p in _out_files:
        if p.exists():
            _shutil.copy2(p, _drive_dir / p.name)
            print(f"[backup] {p.name} -> {_drive_dir / p.name}")
except Exception as exc:
    print("[backup] skipped:", exc)
try:
    from google.colab import files as _colab_files  # type: ignore
    for p in _out_files:
        if p.exists():
            _colab_files.download(str(p))
            print(f"[download] {p.name}")
except Exception as exc:
    print("[download] skipped (not Colab or download blocked):", exc)
